# Kaggle FLUX.2 Klein EVALUATION worker — Checkpoint 4 without Gemini

Runs the hybrid evaluation signals (VLM + SigLIP + OCR) on a separate Kaggle
T4 so the backend can verify Checkpoint 4 with no paid VLM subscription:

    backend (HybridVisionEvaluator, VLM_PROVIDER=kaggle)
        POST {tunnel}/evaluate  { product_images, generated_image(base64),
                                  asset_spec, brand_context }
            |
            |  ngrok tunnel (KAGGLE_EVAL_GATEWAY_URL)
            v
    this worker (own T4) -> SmolVLM2 + SigLIP + RapidOCR -> raw signals JSON

Run order: **Setup → Secrets → Ingest worker → Start worker → Health → Tunnel →
Smoke test.** Copy the printed PUBLIC URL into the backend's
`KAGGLE_EVAL_GATEWAY_URL`. The generation worker keeps its own tunnel +
`KAGGLE_GATEWAY_URL` — two workers, two T4s, two tunnels.

Later, with Gemini access, the backend switches to `VLM_PROVIDER=gemini` with
no changes here.


In [ ]:
# Setup: eval worker deps (VLM via transformers + SigLIP + RapidOCR + server).
# num2words is required by the SmolVLM2 processor (missing it crashes startup).
# pillow pinned <12.0 to keep Kaggle's preinstalled torchvision intact.
!pip install -q transformers "accelerate>=0.26.0" torch rapidocr-onnxruntime \
    num2words "pillow<12.0,>=8.0"
!pip install -q fastapi "uvicorn[standard]" pyngrok requests


In [ ]:
# Load secrets from Kaggle. Add "HF_TOKEN" (for the gated FLUX model — not
# needed for the eval models, harmless to set) and "NGROK_AUTH_TOKEN" under
# Add-ons -> Secrets.
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
NGROK_AUTH_TOKEN = secrets.get_secret("NGROK_AUTH_TOKEN")

print("Secrets loaded:", os.environ["HF_TOKEN"][:8] + "...", NGROK_AUTH_TOKEN[:8] + "...")


In [ ]:
# Ingest the eval worker script from a Kaggle dataset — no pasting.
# Upload notebooks/kaggle_eval_worker.py (from the repo) into any dataset and
# attach it to this notebook; we locate it and copy it into /kaggle/working.
import glob
import shutil

matches = glob.glob("/kaggle/input/**/kaggle_eval_worker.py",recursive=True)
assert matches, (
    "Attach a dataset containing kaggle_eval_worker.py "
    "(repo: notebooks/kaggle_eval_worker.py). Upload it as a dataset and add it to this notebook."
)
src = matches[0]
shutil.copy(src, "/kaggle/working/kaggle_eval_worker.py")
print("Ingested eval worker from:", src)


In [ ]:
import subprocess

log = open("/kaggle/working/eval_worker.log", "w")

proc = subprocess.Popen(
    ["python", "/kaggle/working/kaggle_eval_worker.py"],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("Eval worker PID:", proc.pid)


In [ ]:
# Wait for the eval worker to start AND finish warming (VLM + SigLIP + OCR).
# /health answers immediately; the *_loaded flags flip as models come up.
import time
import requests

for i in range(120):
    try:
        r = requests.get("http://127.0.0.1:8001/health", timeout=5).json()
        print(r)
        if r.get("vlm_loaded") and r.get("siglip_loaded") and r.get("ocr_loaded"):
            break
    except Exception:
        if i % 6 == 0:
            print(f"waiting for eval worker ({i * 5}s)... check eval_worker.log if it stalls")
    time.sleep(5)
else:
    raise RuntimeError("Eval worker did not become ready in time — check eval_worker.log")


In [ ]:
# Expose :8001 to the internet. Copy the PUBLIC URL into the backend's
# KAGGLE_EVAL_GATEWAY_URL env var (keep this cell running while testing).
from pyngrok import ngrok

ngrok.kill()  # clear any stale tunnel from a previous kernel run
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(8001)
print("PUBLIC URL:", tunnel.public_url)
print("Set KAGGLE_EVAL_GATEWAY_URL=" + tunnel.public_url + " on the backend.")


In [ ]:
# Local smoke test through the eval worker API before wiring the backend.
# PRODUCT_IMAGE should be a file in your attached product-image dataset.
import base64
import io
import requests
from PIL import Image

PRODUCT_IMAGE = "/kaggle/input/my-product/p1.webp"  # TODO: your dataset path

# Build a small placeholder "generated ad" to exercise the endpoint.
buf = io.BytesIO()
Image.new("RGB", (512, 512), color=(40, 60, 120)).save(buf, format="PNG")
generated_b64 = base64.b64encode(buf.getvalue()).decode("ascii")

payload = {
    "product_images": [PRODUCT_IMAGE],
    "generated_image": generated_b64,
    "asset_spec": {
        "placement": "ig_feed",
        "width": 1080,
        "height": 1350,
        "generation_prompt": 'Product photography ad. Headline text overlay: "Own the Night".',
    },
    "brand_context": {"name": "Pulse Athletic", "tone": "Bold"},
}

r = requests.post("http://127.0.0.1:8001/evaluate", json=payload, timeout=600)
print("status:", r.status_code)
if r.status_code != 200:
    print("ERROR:", r.text)   # the worker now returns the real cause as JSON
else:
    print(r.json())
